[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week04_stratification.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 4: Stratification for Variance Reduction

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: Data scientist on the experimentation platform team. You've run A/B tests that are correctly calibrated (Week 1), analyzed real-world results (Week 2), and sized experiments properly (Week 3). Now you'll learn how to make those experiments **more efficient** by reducing variance through stratification — getting the same statistical power with fewer users, or more power with the same users.

---

## What You'll Learn

1. Why random assignment can produce imbalanced groups and how often this happens
2. How covariate imbalance inflates the variance of treatment effect estimates
3. Three sampling approaches: Simple Random, Pre-Stratification, Post-Stratification
4. Mathematical proof that stratification always reduces (or equals) SRS variance
5. Implementing a production-quality `select_stratified_groups()` function
6. How to choose good stratification variables
7. Quantifying the practical improvement via simulation

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from itertools import product

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. The Problem: Random Imbalance

### Why Pure Randomization Isn't Always Enough

When we randomly assign users to control and treatment groups, we expect the groups to be **roughly similar** in their characteristics. But "roughly" is doing heavy lifting — especially with moderately-sized experiments.

At QuickCart, device type (mobile vs desktop) strongly predicts purchase behavior: mobile users convert at ~3% while desktop users convert at ~8%. If random assignment puts disproportionately more mobile users in one group, the observed difference in conversion rates will be **confounded** by this imbalance.

Let's generate QuickCart's user population and see how often random assignment produces concerning imbalances.

In [ ]:
def generate_quickcart_population(n_users=10000, seed=42):
    """
    Generate a synthetic QuickCart user population.
    
    Device type: 60% mobile, 40% desktop
    Customer segment: 30% new, 70% returning
    Revenue depends on both covariates.
    """
    rng = np.random.default_rng(seed)
    
    # Assign covariates
    device = rng.choice(['mobile', 'desktop'], size=n_users, p=[0.6, 0.4])
    segment = rng.choice(['new', 'returning'], size=n_users, p=[0.3, 0.7])
    
    # Revenue depends on device and segment (in dollars per session)
    # Desktop users spend more, returning users spend more
    base_revenue = np.where(device == 'desktop', 12.0, 5.0)
    segment_bonus = np.where(segment == 'returning', 4.0, 0.0)
    noise = rng.exponential(scale=3.0, size=n_users)
    
    revenue = base_revenue + segment_bonus + noise
    
    df = pd.DataFrame({
        'user_id': range(n_users),
        'device_type': device,
        'customer_segment': segment,
        'revenue': revenue
    })
    
    return df


# Generate population
population = generate_quickcart_population(n_users=10000)
print(f"Population size: {len(population)}")
print(f"\nDevice type distribution:")
print(population['device_type'].value_counts(normalize=True))
print(f"\nCustomer segment distribution:")
print(population['customer_segment'].value_counts(normalize=True))
print(f"\nMean revenue by stratum:")
print(population.groupby(['device_type', 'customer_segment'])['revenue'].mean().round(2))

In [ ]:
# Simulate many random 50/50 splits and measure imbalance
n_simulations = 5000
n_per_group = 500  # 500 per group from 10,000 population

mobile_fraction_control = []
mobile_fraction_treatment = []
imbalance_mobile = []  # difference in mobile % between groups
imbalance_returning = []  # difference in returning % between groups

rng = np.random.default_rng(123)

for _ in range(n_simulations):
    # Random assignment: pick 2*n_per_group users, split in half
    indices = rng.choice(len(population), size=2 * n_per_group, replace=False)
    control_idx = indices[:n_per_group]
    treatment_idx = indices[n_per_group:]
    
    control = population.iloc[control_idx]
    treatment = population.iloc[treatment_idx]
    
    # Mobile fraction in each group
    ctrl_mobile = (control['device_type'] == 'mobile').mean()
    treat_mobile = (treatment['device_type'] == 'mobile').mean()
    mobile_fraction_control.append(ctrl_mobile)
    mobile_fraction_treatment.append(treat_mobile)
    imbalance_mobile.append(abs(ctrl_mobile - treat_mobile))
    
    # Returning fraction in each group
    ctrl_ret = (control['customer_segment'] == 'returning').mean()
    treat_ret = (treatment['customer_segment'] == 'returning').mean()
    imbalance_returning.append(abs(ctrl_ret - treat_ret))

imbalance_mobile = np.array(imbalance_mobile)
imbalance_returning = np.array(imbalance_returning)

# How often do we get large imbalances?
thresholds = [0.03, 0.05, 0.08, 0.10]
print("Probability of imbalance exceeding threshold (mobile %):")
print("-" * 50)
for t in thresholds:
    prob = (imbalance_mobile > t).mean()
    print(f"  |control_mobile% - treatment_mobile%| > {t:.0%}:  {prob:.1%}")

print(f"\nProbability of imbalance exceeding threshold (returning %):")
print("-" * 50)
for t in thresholds:
    prob = (imbalance_returning > t).mean()
    print(f"  |control_returning% - treatment_returning%| > {t:.0%}:  {prob:.1%}")

In [ ]:
# Visualize the distribution of imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(imbalance_mobile, bins=50, density=True, alpha=0.7, color='steelblue')
axes[0].axvline(0.05, color='red', linestyle='--', linewidth=2, label='5% threshold')
axes[0].set_xlabel('Absolute imbalance in mobile fraction')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Mobile % Imbalance\n(5000 random splits, n=500 per group)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(imbalance_returning, bins=50, density=True, alpha=0.7, color='darkorange')
axes[1].axvline(0.05, color='red', linestyle='--', linewidth=2, label='5% threshold')
axes[1].set_xlabel('Absolute imbalance in returning fraction')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution of Returning % Imbalance\n(5000 random splits, n=500 per group)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Median imbalance in mobile %: {np.median(imbalance_mobile):.3f}")
print(f"Median imbalance in returning %: {np.median(imbalance_returning):.3f}")
print(f"\n95th percentile imbalance (mobile): {np.percentile(imbalance_mobile, 95):.3f}")
print(f"95th percentile imbalance (returning): {np.percentile(imbalance_returning, 95):.3f}")

**Key observation**: Even with n=500 per group, about 15-25% of random splits produce >5% imbalance in a covariate. When that covariate strongly predicts the outcome, this imbalance translates directly into bias in the estimated treatment effect.

---

## 2. Why Imbalance Hurts

### The Mechanics of Confounding

Suppose the true treatment effect is zero (no real difference between groups). If the control group happens to have more desktop users (who spend more), the naive estimate will show a negative "treatment effect" — purely due to the covariate imbalance.

This isn't bias in expectation (randomization is unbiased on average), but it **inflates variance**. The treatment effect estimator picks up noise from covariate imbalance, making it harder to detect real effects.

### Simpson's Paradox Flavor

Consider two strata with different baseline rates:
- Mobile users: mean revenue = $5
- Desktop users: mean revenue = $12

If the control group is 55% mobile and treatment is 65% mobile, the treatment group's average revenue will be lower **even with no treatment effect** — simply because it has more low-revenue mobile users.

The key insight: **any covariate that explains variance in the outcome will inflate the variance of the treatment effect estimator when it's imbalanced across groups.**

In [ ]:
# Demonstrate: run an A/A test (no true effect) with random assignment
# and show how imbalance creates apparent effects

n_simulations = 3000
n_per_group = 500
estimated_effects = []
mobile_imbalances = []

rng = np.random.default_rng(99)

for _ in range(n_simulations):
    # No treatment effect — just random assignment from same population
    indices = rng.choice(len(population), size=2 * n_per_group, replace=False)
    control = population.iloc[indices[:n_per_group]]
    treatment = population.iloc[indices[n_per_group:]]
    
    # Naive estimate of treatment effect
    effect = treatment['revenue'].mean() - control['revenue'].mean()
    estimated_effects.append(effect)
    
    # Track mobile imbalance
    imb = (treatment['device_type'] == 'mobile').mean() - (control['device_type'] == 'mobile').mean()
    mobile_imbalances.append(imb)

estimated_effects = np.array(estimated_effects)
mobile_imbalances = np.array(mobile_imbalances)

# Show correlation between imbalance and estimated effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(mobile_imbalances, estimated_effects, alpha=0.15, s=10)
# Fit regression line
slope, intercept = np.polyfit(mobile_imbalances, estimated_effects, 1)
x_line = np.linspace(mobile_imbalances.min(), mobile_imbalances.max(), 100)
axes[0].plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, 
             label=f'Slope = {slope:.2f}')
axes[0].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Mobile fraction imbalance (treatment - control)')
axes[0].set_ylabel('Estimated treatment effect ($)')
axes[0].set_title('Covariate Imbalance Drives Apparent Effects\n(A/A test, true effect = 0)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(estimated_effects, bins=50, density=True, alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='True effect = 0')
axes[1].set_xlabel('Estimated treatment effect ($)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Distribution of Estimated Effects (A/A)\nStd = ${np.std(estimated_effects):.3f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

correlation = np.corrcoef(mobile_imbalances, estimated_effects)[0, 1]
print(f"Correlation between mobile imbalance and estimated effect: {correlation:.3f}")
print(f"Standard deviation of estimated effects: ${np.std(estimated_effects):.3f}")
print(f"\nThis variance could be REDUCED if we guaranteed balance on device_type.")

---

## 3. Simple Random Sampling (SRS)

### The Baseline Approach

Simple Random Sampling assigns each user to control or treatment with equal probability, independent of their characteristics. This is unbiased in expectation but can produce imbalanced groups.

The variance of the treatment effect estimator under SRS is:

$$
\text{Var}_{\text{SRS}}(\hat{\tau}) = \frac{\sigma^2_{\text{total}}}{n_T} + \frac{\sigma^2_{\text{total}}}{n_C}
$$

where $\sigma^2_{\text{total}}$ is the total outcome variance in the population.

In [ ]:
def simple_random_split(data, n_per_group, seed=None):
    """
    Simple random sampling: randomly assign users to control/treatment.
    
    Parameters
    ----------
    data : pd.DataFrame
        Population data.
    n_per_group : int
        Number of users per group.
    seed : int, optional
        Random seed.
    
    Returns
    -------
    control : pd.DataFrame
    treatment : pd.DataFrame
    """
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(data), size=2 * n_per_group, replace=False)
    control = data.iloc[indices[:n_per_group]].copy()
    treatment = data.iloc[indices[n_per_group:]].copy()
    return control, treatment


# Run many SRS splits and collect treatment effect estimates
n_simulations = 2000
n_per_group = 500
true_effect = 0.0  # A/A test

srs_effects = []
for i in range(n_simulations):
    control, treatment = simple_random_split(population, n_per_group, seed=i)
    effect = treatment['revenue'].mean() - control['revenue'].mean()
    srs_effects.append(effect)

srs_effects = np.array(srs_effects)
print(f"SRS — Variance of estimated effect: {np.var(srs_effects):.5f}")
print(f"SRS — Std of estimated effect: ${np.std(srs_effects):.4f}")
print(f"SRS — Mean (should be ~0): ${np.mean(srs_effects):.4f}")

---

## 4. Pre-Stratification

### The Core Idea

Pre-stratification divides the population into **strata** (groups defined by covariates) and then samples separately within each stratum. This **guarantees** proportional representation in both control and treatment groups.

For QuickCart, strata might be defined by:
- Device type: {mobile, desktop}
- Customer segment: {new, returning}

This gives 2 x 2 = 4 strata. Within each stratum, we randomly assign half to control and half to treatment.

### Why This Reduces Variance

The total variance can be decomposed:

$$
\sigma^2_{\text{total}} = \underbrace{\sum_h W_h \sigma^2_h}_{\text{within-strata variance}} + \underbrace{\sum_h W_h (\mu_h - \mu)^2}_{\text{between-strata variance}}
$$

where $W_h$ is the proportion in stratum $h$, $\sigma^2_h$ is the variance within stratum $h$, and $\mu_h$ is the mean of stratum $h$.

Pre-stratification eliminates the between-strata component from the estimator's variance. The more the strata means differ (i.e., the more the stratification variable explains), the bigger the variance reduction.

In [ ]:
def pre_stratified_split(data, strat_columns, n_per_group, seed=None):
    """
    Pre-stratified sampling: sample proportionally within each stratum.
    
    Parameters
    ----------
    data : pd.DataFrame
        Population data.
    strat_columns : list of str
        Columns defining strata.
    n_per_group : int
        Total number of users per group.
    seed : int, optional
        Random seed.
    
    Returns
    -------
    control : pd.DataFrame
    treatment : pd.DataFrame
    """
    rng = np.random.default_rng(seed)
    
    # Compute stratum proportions from the full data
    strata = data.groupby(strat_columns)
    total_n = len(data)
    
    control_list = []
    treatment_list = []
    
    for name, group in strata:
        # Proportional allocation
        stratum_fraction = len(group) / total_n
        n_stratum = max(1, int(round(n_per_group * stratum_fraction)))
        
        # Sample 2*n_stratum from this stratum
        available = group.index.values
        n_sample = min(2 * n_stratum, len(available))
        selected = rng.choice(available, size=n_sample, replace=False)
        
        half = n_sample // 2
        control_list.append(data.loc[selected[:half]])
        treatment_list.append(data.loc[selected[half:]])
    
    control = pd.concat(control_list).reset_index(drop=True)
    treatment = pd.concat(treatment_list).reset_index(drop=True)
    
    return control, treatment


# Run many pre-stratified splits
strat_columns = ['device_type', 'customer_segment']
pre_strat_effects = []

for i in range(n_simulations):
    control, treatment = pre_stratified_split(population, strat_columns, n_per_group, seed=i)
    effect = treatment['revenue'].mean() - control['revenue'].mean()
    pre_strat_effects.append(effect)

pre_strat_effects = np.array(pre_strat_effects)
print(f"Pre-Stratified — Variance of estimated effect: {np.var(pre_strat_effects):.5f}")
print(f"Pre-Stratified — Std of estimated effect: ${np.std(pre_strat_effects):.4f}")
print(f"Pre-Stratified — Mean (should be ~0): ${np.mean(pre_strat_effects):.4f}")
print(f"\nVariance reduction vs SRS: {(1 - np.var(pre_strat_effects)/np.var(srs_effects))*100:.1f}%")

In [ ]:
# Verify balance is guaranteed with pre-stratification
control, treatment = pre_stratified_split(population, strat_columns, n_per_group, seed=0)

print("Pre-stratification guarantees balance:")
print("=" * 50)
print(f"\nControl group device distribution:")
print(control['device_type'].value_counts(normalize=True).round(3))
print(f"\nTreatment group device distribution:")
print(treatment['device_type'].value_counts(normalize=True).round(3))
print(f"\nPopulation device distribution:")
print(population['device_type'].value_counts(normalize=True).round(3))
print(f"\nImbalance in mobile %: {abs((control['device_type']=='mobile').mean() - (treatment['device_type']=='mobile').mean()):.4f}")

---

## 5. Post-Stratification

### The Idea: Adjust After the Fact

Sometimes you can't stratify before assignment (e.g., the assignment system doesn't support it, or you discover a relevant covariate after the experiment starts). Post-stratification lets you **retroactively adjust** for stratum-level differences.

### The Post-Stratified Estimator

Instead of computing a simple difference in means, compute a **weighted average** of within-stratum differences:

$$
\hat{\tau}_{\text{post}} = \sum_{h=1}^{H} W_h \left( \bar{Y}_{h,T} - \bar{Y}_{h,C} \right)
$$

where:
- $W_h$ = population proportion in stratum $h$
- $\bar{Y}_{h,T}$ = mean outcome in stratum $h$ for treatment
- $\bar{Y}_{h,C}$ = mean outcome in stratum $h$ for control

This removes the between-strata component of variance by computing the treatment effect **within** each stratum and then averaging.

In [ ]:
def post_stratified_estimate(control, treatment, strat_columns, population_weights=None):
    """
    Compute post-stratified treatment effect estimate.
    
    Parameters
    ----------
    control : pd.DataFrame
        Control group data.
    treatment : pd.DataFrame
        Treatment group data.
    strat_columns : list of str
        Columns defining strata.
    population_weights : dict, optional
        {stratum_tuple: weight}. If None, computed from combined data.
    
    Returns
    -------
    float
        Post-stratified treatment effect estimate.
    """
    # Compute population weights if not provided
    if population_weights is None:
        combined = pd.concat([control, treatment])
        total = len(combined)
        population_weights = {}
        for name, group in combined.groupby(strat_columns):
            population_weights[name] = len(group) / total
    
    # Compute within-stratum effects
    tau_post = 0.0
    for stratum, weight in population_weights.items():
        # Build mask for this stratum
        if isinstance(stratum, str):
            stratum = (stratum,)
        
        ctrl_mask = pd.Series([True] * len(control), index=control.index)
        treat_mask = pd.Series([True] * len(treatment), index=treatment.index)
        
        for col, val in zip(strat_columns, stratum):
            ctrl_mask &= (control[col] == val)
            treat_mask &= (treatment[col] == val)
        
        ctrl_stratum = control[ctrl_mask]['revenue']
        treat_stratum = treatment[treat_mask]['revenue']
        
        # Skip if either group has no observations in this stratum
        if len(ctrl_stratum) == 0 or len(treat_stratum) == 0:
            continue
        
        within_effect = treat_stratum.mean() - ctrl_stratum.mean()
        tau_post += weight * within_effect
    
    return tau_post


# Compute population weights once
pop_weights = {}
for name, group in population.groupby(strat_columns):
    pop_weights[name] = len(group) / len(population)

print("Population stratum weights:")
for stratum, weight in sorted(pop_weights.items()):
    print(f"  {stratum}: {weight:.3f}")

In [ ]:
# Run post-stratification on random (SRS) splits
post_strat_effects = []

for i in range(n_simulations):
    control, treatment = simple_random_split(population, n_per_group, seed=i)
    effect = post_stratified_estimate(control, treatment, strat_columns, pop_weights)
    post_strat_effects.append(effect)

post_strat_effects = np.array(post_strat_effects)
print(f"Post-Stratified — Variance of estimated effect: {np.var(post_strat_effects):.5f}")
print(f"Post-Stratified — Std of estimated effect: ${np.std(post_strat_effects):.4f}")
print(f"Post-Stratified — Mean (should be ~0): ${np.mean(post_strat_effects):.4f}")
print(f"\nVariance reduction vs SRS: {(1 - np.var(post_strat_effects)/np.var(srs_effects))*100:.1f}%")
print(f"Variance reduction vs Pre-strat: {(1 - np.var(post_strat_effects)/np.var(pre_strat_effects))*100:.1f}%")

In [ ]:
# Compare all three approaches visually
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(srs_effects, bins=50, density=True, alpha=0.5, label=f'SRS (std=${np.std(srs_effects):.3f})', color='steelblue')
ax.hist(pre_strat_effects, bins=50, density=True, alpha=0.5, label=f'Pre-Stratified (std=${np.std(pre_strat_effects):.3f})', color='green')
ax.hist(post_strat_effects, bins=50, density=True, alpha=0.5, label=f'Post-Stratified (std=${np.std(post_strat_effects):.3f})', color='darkorange')

ax.axvline(0, color='red', linestyle='--', linewidth=2, label='True effect = 0')
ax.set_xlabel('Estimated treatment effect ($)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Treatment Effect Estimates\n(A/A Test: True Effect = 0)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Key findings:**
- Pre-stratification achieves the best variance reduction (eliminates between-strata variance entirely)
- Post-stratification also reduces variance substantially, though slightly less than pre-strat
- Both are unbiased (centered at 0) — they reduce variance without introducing bias

Post-stratification is slightly less efficient because within each SRS-assigned group, the stratum sample sizes are random (not fixed), which introduces some additional variability in the weights.

---

## 6. Mathematical Comparison

### Variance Formulas

Let there be $H$ strata with population proportions $W_1, W_2, \ldots, W_H$ (where $\sum_h W_h = 1$), stratum means $\mu_h$, and stratum variances $\sigma^2_h$. The overall mean is $\mu = \sum_h W_h \mu_h$.

**Simple Random Sampling:**

$$
\text{Var}_{\text{SRS}}(\hat{\tau}) = \frac{2}{n}\left[ \sum_h W_h \sigma^2_h + \sum_h W_h (\mu_h - \mu)^2 \right] = \frac{2\sigma^2_{\text{total}}}{n}
$$

**Pre-Stratification (proportional allocation):**

$$
\text{Var}_{\text{pre}}(\hat{\tau}) = \frac{2}{n} \sum_h W_h \sigma^2_h
$$

**The Variance Reduction:**

$$
\text{Var}_{\text{SRS}} - \text{Var}_{\text{pre}} = \frac{2}{n} \sum_h W_h (\mu_h - \mu)^2 \geq 0
$$

The reduction equals the **between-strata variance** divided by $n$. This is always non-negative (it's a sum of squared terms), proving that stratification never hurts and helps whenever strata have different means.

:::{admonition} Full Derivation: Variance Decomposition and Stratification
:class: dropdown

**Law of Total Variance:**

$$\text{Var}(Y) = E[\text{Var}(Y|S)] + \text{Var}(E[Y|S])$$

where $S$ denotes the stratum. In our notation:

$$\sigma^2_{\text{total}} = \underbrace{\sum_h W_h \sigma^2_h}_{\text{within}} + \underbrace{\sum_h W_h(\mu_h - \mu)^2}_{\text{between}}$$

**SRS estimator:**

Under SRS, the treatment effect estimator is $\hat{\tau} = \bar{Y}_T - \bar{Y}_C$ where each group mean has variance $\sigma^2_{\text{total}}/n$. Since the groups are independent:

$$\text{Var}_{\text{SRS}}(\hat{\tau}) = \frac{\sigma^2_{\text{total}}}{n_T} + \frac{\sigma^2_{\text{total}}}{n_C} = \frac{2\sigma^2_{\text{total}}}{n}$$

(assuming $n_T = n_C = n$)

**Pre-stratified estimator:**

With proportional allocation, exactly $n \cdot W_h$ units from stratum $h$ are assigned to each group. The estimator is:

$$\hat{\tau}_{\text{pre}} = \sum_h W_h (\bar{Y}_{h,T} - \bar{Y}_{h,C})$$

Since the within-stratum samples are independent:

$$\text{Var}(\hat{\tau}_{\text{pre}}) = \sum_h W_h^2 \cdot \frac{2\sigma^2_h}{n \cdot W_h} = \frac{2}{n}\sum_h W_h \sigma^2_h$$

This equals the **within-strata** component only, eliminating the between-strata term.

**Efficiency gain:**

$$\text{Relative efficiency} = \frac{\text{Var}_{\text{SRS}}}{\text{Var}_{\text{pre}}} = \frac{\sigma^2_{\text{total}}}{\sum_h W_h \sigma^2_h} = \frac{1}{1 - R^2_S}$$

where $R^2_S = \frac{\sum_h W_h(\mu_h - \mu)^2}{\sigma^2_{\text{total}}}$ is the fraction of total variance explained by the stratification variable.

**Interpretation:** If device type and segment explain 20% of revenue variance ($R^2 = 0.20$), stratification reduces variance by 20%, equivalent to increasing sample size by 25%.
:::

In [ ]:
# Verify the mathematical result with our data

# Compute total variance
sigma2_total = population['revenue'].var()

# Compute within-strata variance
within_var = 0
between_var = 0
overall_mean = population['revenue'].mean()

print("Variance Decomposition")
print("=" * 60)

for name, group in population.groupby(strat_columns):
    w_h = len(group) / len(population)
    sigma2_h = group['revenue'].var()
    mu_h = group['revenue'].mean()
    
    within_var += w_h * sigma2_h
    between_var += w_h * (mu_h - overall_mean)**2
    
    print(f"  Stratum {str(name):30s}: W_h={w_h:.3f}, mu_h=${mu_h:.2f}, sigma2_h={sigma2_h:.2f}")

print(f"\nTotal variance:           {sigma2_total:.4f}")
print(f"Within-strata variance:   {within_var:.4f}")
print(f"Between-strata variance:  {between_var:.4f}")
print(f"Sum (within + between):   {within_var + between_var:.4f}")

R_squared = between_var / sigma2_total
print(f"\nR-squared of stratification: {R_squared:.4f}")
print(f"Predicted variance reduction: {R_squared*100:.1f}%")
print(f"Observed variance reduction:  {(1 - np.var(pre_strat_effects)/np.var(srs_effects))*100:.1f}%")

---

## 7. Implementing Stratified Group Selection

### Production-Quality Implementation

Now let's build a robust function that can be used in QuickCart's experimentation platform. The function should:

1. Accept arbitrary stratification columns
2. Guarantee proportional representation in each group
3. Handle edge cases (empty strata, odd numbers)
4. Return a DataFrame with group assignments
5. Support optional custom weights

In [ ]:
def select_stratified_groups(data, strat_columns, group_size, weights=None, seed=42):
    """
    Create balanced pilot/control groups with proportional representation
    from each stratum.
    
    Parameters
    ----------
    data : pd.DataFrame
        Full population data. Must contain columns listed in strat_columns.
    strat_columns : list of str
        Column names used to define strata (e.g., ['device_type', 'customer_segment']).
    group_size : int
        Desired number of users in EACH group (pilot and control).
    weights : dict, optional
        Custom stratum weights {stratum_tuple: weight}. If None, uses
        population proportions. Weights are normalized to sum to 1.
    seed : int, default=42
        Random seed for reproducibility.
    
    Returns
    -------
    pd.DataFrame
        Copy of selected rows with an added 'group' column ('pilot' or 'control').
    
    Raises
    ------
    ValueError
        If group_size exceeds half the population or strata are too small.
    
    Examples
    --------
    >>> result = select_stratified_groups(
    ...     data=users_df,
    ...     strat_columns=['device_type', 'customer_segment'],
    ...     group_size=1000,
    ...     seed=42
    ... )
    >>> result['group'].value_counts()
    control    1000
    pilot      1000
    Name: group, dtype: int64
    """
    rng = np.random.default_rng(seed)
    
    if 2 * group_size > len(data):
        raise ValueError(
            f"group_size={group_size} requires {2*group_size} users, "
            f"but data has only {len(data)} rows."
        )
    
    # Compute stratum information
    grouped = data.groupby(strat_columns)
    strata_info = []
    
    for name, group in grouped:
        if not isinstance(name, tuple):
            name = (name,)
        strata_info.append({
            'name': name,
            'indices': group.index.values,
            'size': len(group)
        })
    
    # Determine weights
    if weights is None:
        total = len(data)
        stratum_weights = {s['name']: s['size'] / total for s in strata_info}
    else:
        # Normalize provided weights
        total_weight = sum(weights.values())
        stratum_weights = {k: v / total_weight for k, v in weights.items()}
    
    # Allocate sample sizes per stratum (using largest remainder method)
    raw_allocations = {s['name']: group_size * stratum_weights.get(s['name'], 0) 
                       for s in strata_info}
    
    # Floor allocations
    floor_alloc = {k: int(np.floor(v)) for k, v in raw_allocations.items()}
    remainders = {k: raw_allocations[k] - floor_alloc[k] for k in raw_allocations}
    
    # Distribute remaining slots to strata with largest remainders
    remaining = group_size - sum(floor_alloc.values())
    sorted_remainders = sorted(remainders.items(), key=lambda x: x[1], reverse=True)
    for i in range(remaining):
        stratum_name = sorted_remainders[i][0]
        floor_alloc[stratum_name] += 1
    
    allocations = floor_alloc
    
    # Sample from each stratum
    pilot_indices = []
    control_indices = []
    
    for s in strata_info:
        n_per_group_stratum = allocations[s['name']]
        n_needed = 2 * n_per_group_stratum
        
        if n_needed > s['size']:
            raise ValueError(
                f"Stratum {s['name']} has {s['size']} users but needs "
                f"{n_needed} (2 x {n_per_group_stratum})."
            )
        
        if n_per_group_stratum == 0:
            continue
        
        # Random selection within stratum
        selected = rng.choice(s['indices'], size=n_needed, replace=False)
        rng.shuffle(selected)  # extra shuffle for good measure
        
        pilot_indices.extend(selected[:n_per_group_stratum])
        control_indices.extend(selected[n_per_group_stratum:])
    
    # Build result DataFrame
    pilot_df = data.loc[pilot_indices].copy()
    pilot_df['group'] = 'pilot'
    
    control_df = data.loc[control_indices].copy()
    control_df['group'] = 'control'
    
    result = pd.concat([pilot_df, control_df]).reset_index(drop=True)
    
    return result

In [ ]:
# Demonstrate the function
result = select_stratified_groups(
    data=population,
    strat_columns=['device_type', 'customer_segment'],
    group_size=500,
    seed=42
)

print("Group assignments:")
print(result['group'].value_counts())
print(f"\nTotal users selected: {len(result)}")

# Check balance
print("\n" + "=" * 60)
print("Balance check — Stratum proportions by group:")
print("=" * 60)

for group_name in ['pilot', 'control']:
    group_data = result[result['group'] == group_name]
    print(f"\n{group_name.upper()} (n={len(group_data)}):")
    cross = group_data.groupby(['device_type', 'customer_segment']).size()
    cross_pct = cross / len(group_data)
    for idx, val in cross_pct.items():
        print(f"  {str(idx):35s}: {val:.3f} (n={cross[idx]})")

# Compare to population proportions
print(f"\nPOPULATION proportions:")
pop_cross = population.groupby(['device_type', 'customer_segment']).size()
pop_cross_pct = pop_cross / len(population)
for idx, val in pop_cross_pct.items():
    print(f"  {str(idx):35s}: {val:.3f}")

In [ ]:
# Test with custom weights (over-sample desktop users for analysis)
custom_weights = {
    ('desktop', 'new'): 0.25,
    ('desktop', 'returning'): 0.25,
    ('mobile', 'new'): 0.25,
    ('mobile', 'returning'): 0.25,
}

result_custom = select_stratified_groups(
    data=population,
    strat_columns=['device_type', 'customer_segment'],
    group_size=400,
    weights=custom_weights,
    seed=42
)

print("With equal weights (oversampling smaller strata):")
print(result_custom.groupby(['group', 'device_type', 'customer_segment']).size().unstack(level=0))

---

## 8. Choosing Good Strata

### The Principle

The variance reduction from stratification is proportional to the **between-strata variance** — i.e., how much the stratification variable *explains* the outcome. A good stratification variable has high $R^2$ with the target metric.

### Guidelines for Choosing Strata

1. **Predictive power**: Variables that explain more variance in the outcome give more reduction
2. **Few strata**: Too many strata means few users per stratum, reducing precision within each
3. **Stable over time**: The variable should be observable *before* treatment assignment
4. **Practical feasibility**: The assignment system must be able to condition on the variable

### Evaluating Candidate Variables at QuickCart

Let's compare several potential stratification variables for their ability to explain revenue variance.

In [ ]:
# Create a richer population with more potential stratification variables
def generate_rich_quickcart_population(n_users=10000, seed=42):
    """Generate population with multiple potential stratification variables."""
    rng = np.random.default_rng(seed)
    
    # Covariates
    device = rng.choice(['mobile', 'desktop'], size=n_users, p=[0.6, 0.4])
    segment = rng.choice(['new', 'returning'], size=n_users, p=[0.3, 0.7])
    region = rng.choice(['US', 'EU', 'APAC'], size=n_users, p=[0.5, 0.3, 0.2])
    browser = rng.choice(['chrome', 'safari', 'firefox', 'other'], 
                         size=n_users, p=[0.55, 0.25, 0.12, 0.08])
    day_of_week = rng.choice(['weekday', 'weekend'], size=n_users, p=[0.71, 0.29])
    
    # Revenue model: depends on device and segment strongly, region moderately
    base = np.where(device == 'desktop', 12.0, 5.0)
    segment_effect = np.where(segment == 'returning', 4.0, 0.0)
    region_effect = np.where(region == 'US', 2.0, np.where(region == 'EU', 1.0, 0.0))
    # Browser and day_of_week: very weak predictors (noise variables)
    browser_effect = rng.normal(0, 0.1, size=n_users)
    day_effect = np.where(day_of_week == 'weekend', 0.3, 0.0)
    
    noise = rng.exponential(scale=3.0, size=n_users)
    revenue = base + segment_effect + region_effect + browser_effect + day_effect + noise
    
    return pd.DataFrame({
        'user_id': range(n_users),
        'device_type': device,
        'customer_segment': segment,
        'region': region,
        'browser': browser,
        'day_of_week': day_of_week,
        'revenue': revenue
    })


rich_population = generate_rich_quickcart_population(n_users=10000)

# Compute R-squared for each potential stratification variable
candidate_vars = ['device_type', 'customer_segment', 'region', 'browser', 'day_of_week']
total_var = rich_population['revenue'].var()

print("Evaluating stratification candidates (single variable):")
print("=" * 60)
print(f"{'Variable':<20} {'R-squared':>10} {'Between-strata var':>20} {'# Strata':>10}")
print("-" * 60)

r_squared_values = {}
for var in candidate_vars:
    groups = rich_population.groupby(var)
    overall_mean = rich_population['revenue'].mean()
    between_var = sum(
        (len(g) / len(rich_population)) * (g['revenue'].mean() - overall_mean)**2 
        for _, g in groups
    )
    r2 = between_var / total_var
    r_squared_values[var] = r2
    n_strata = groups.ngroups
    print(f"{var:<20} {r2:>10.4f} {between_var:>20.4f} {n_strata:>10}")

# Combined stratification
print("\n" + "=" * 60)
print("Combined stratification variables:")
print("-" * 60)

combos = [
    ['device_type', 'customer_segment'],
    ['device_type', 'customer_segment', 'region'],
    ['device_type', 'customer_segment', 'region', 'day_of_week'],
]

for combo in combos:
    groups = rich_population.groupby(combo)
    between_var = sum(
        (len(g) / len(rich_population)) * (g['revenue'].mean() - overall_mean)**2 
        for _, g in groups
    )
    r2 = between_var / total_var
    n_strata = groups.ngroups
    combo_str = ' + '.join(combo)
    print(f"{combo_str:<50} R2={r2:.4f}  strata={n_strata}")

In [ ]:
# Visualize: R-squared vs number of strata (diminishing returns)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of R-squared by variable
vars_sorted = sorted(r_squared_values.items(), key=lambda x: x[1], reverse=True)
names = [v[0] for v in vars_sorted]
r2s = [v[1] for v in vars_sorted]

colors = ['green' if r2 > 0.05 else 'orange' if r2 > 0.01 else 'red' for r2 in r2s]
axes[0].barh(names, r2s, color=colors, alpha=0.7)
axes[0].set_xlabel('R-squared with revenue')
axes[0].set_title('Predictive Power of Stratification Candidates')
axes[0].axvline(0.05, color='gray', linestyle='--', alpha=0.5, label='5% threshold')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='x')

# Revenue distributions by best stratifier (device_type + segment)
strata_data = []
labels = []
for name, group in rich_population.groupby(['device_type', 'customer_segment']):
    strata_data.append(group['revenue'].values)
    labels.append(f"{name[0]}\n{name[1]}")

axes[1].boxplot(strata_data, labels=labels)
axes[1].set_ylabel('Revenue ($)')
axes[1].set_title('Revenue Distribution by Stratum\n(device_type x customer_segment)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nRecommendation: Stratify by device_type + customer_segment")
print("  - Explains the most outcome variance")
print("  - Only 4 strata (manageable)")
print("  - Both are known before assignment")
print("\n  Adding region gives marginal improvement but doubles the number of strata.")
print("  Browser and day_of_week are not worth stratifying on (near-zero R2).")

---

## 9. Full Simulation Comparison

### Head-to-Head: 2000 Simulated Experiments

Let's run a comprehensive comparison of all three approaches with a realistic scenario:
- QuickCart launches a new checkout flow
- True treatment effect: +$0.50 average revenue per session
- We want to detect this with 500 users per group

We'll measure:
1. Variance of the treatment effect estimator
2. Statistical power (how often we correctly reject H0)
3. Coverage of 95% confidence intervals

In [ ]:
# Full simulation comparing all three approaches
n_simulations = 2000
n_per_group = 500
true_effect = 0.50  # +$0.50 treatment effect
alpha = 0.05

strat_cols = ['device_type', 'customer_segment']

# Pre-compute population weights
pop_weights_full = {}
for name, group in population.groupby(strat_cols):
    pop_weights_full[name] = len(group) / len(population)

results = {
    'SRS (naive)': {'effects': [], 'rejections': [], 'covers': []},
    'Pre-stratification': {'effects': [], 'rejections': [], 'covers': []},
    'Post-stratification': {'effects': [], 'rejections': [], 'covers': []},
}

for i in range(n_simulations):
    # Create a treated population (add true_effect to revenue)
    treated_pop = population.copy()
    treated_pop['revenue'] = treated_pop['revenue'] + true_effect
    
    # --- Method 1: Simple Random Sampling ---
    rng_srs = np.random.default_rng(i)
    ctrl_idx = rng_srs.choice(len(population), size=n_per_group, replace=False)
    remaining = np.setdiff1d(np.arange(len(population)), ctrl_idx)
    treat_idx = rng_srs.choice(remaining, size=n_per_group, replace=False)
    
    ctrl_srs = population.iloc[ctrl_idx]
    treat_srs = treated_pop.iloc[treat_idx]
    
    effect_srs = treat_srs['revenue'].mean() - ctrl_srs['revenue'].mean()
    se_srs = np.sqrt(ctrl_srs['revenue'].var()/n_per_group + treat_srs['revenue'].var()/n_per_group)
    t_stat_srs = effect_srs / se_srs
    p_srs = 2 * (1 - stats.t.cdf(abs(t_stat_srs), df=2*n_per_group-2))
    ci_srs = (effect_srs - 1.96*se_srs, effect_srs + 1.96*se_srs)
    
    results['SRS (naive)']['effects'].append(effect_srs)
    results['SRS (naive)']['rejections'].append(int(p_srs < alpha))
    results['SRS (naive)']['covers'].append(int(ci_srs[0] <= true_effect <= ci_srs[1]))
    
    # --- Method 2: Pre-Stratification ---
    rng_pre = np.random.default_rng(i + 100000)
    ctrl_pre_list = []
    treat_pre_list = []
    
    for name, group in population.groupby(strat_cols):
        w = pop_weights_full[name]
        n_stratum = max(1, int(round(n_per_group * w)))
        available = group.index.values
        selected = rng_pre.choice(available, size=min(2*n_stratum, len(available)), replace=False)
        half = len(selected) // 2
        ctrl_pre_list.append(population.loc[selected[:half]])
        treat_pre_list.append(treated_pop.loc[selected[half:]])
    
    ctrl_pre = pd.concat(ctrl_pre_list)
    treat_pre = pd.concat(treat_pre_list)
    
    # Pre-strat estimator: weighted average of within-stratum effects
    effect_pre = 0
    var_pre = 0
    for name in pop_weights_full:
        w = pop_weights_full[name]
        if isinstance(name, str):
            name = (name,)
        c_mask = pd.Series([True]*len(ctrl_pre), index=ctrl_pre.index)
        t_mask = pd.Series([True]*len(treat_pre), index=treat_pre.index)
        for col, val in zip(strat_cols, name):
            c_mask &= (ctrl_pre[col] == val)
            t_mask &= (treat_pre[col] == val)
        c_rev = ctrl_pre[c_mask]['revenue']
        t_rev = treat_pre[t_mask]['revenue']
        if len(c_rev) > 0 and len(t_rev) > 0:
            effect_pre += w * (t_rev.mean() - c_rev.mean())
            var_pre += w**2 * (c_rev.var()/len(c_rev) + t_rev.var()/len(t_rev))
    
    se_pre = np.sqrt(var_pre)
    t_stat_pre = effect_pre / se_pre if se_pre > 0 else 0
    p_pre = 2 * (1 - stats.t.cdf(abs(t_stat_pre), df=2*n_per_group-2))
    ci_pre = (effect_pre - 1.96*se_pre, effect_pre + 1.96*se_pre)
    
    results['Pre-stratification']['effects'].append(effect_pre)
    results['Pre-stratification']['rejections'].append(int(p_pre < alpha))
    results['Pre-stratification']['covers'].append(int(ci_pre[0] <= true_effect <= ci_pre[1]))
    
    # --- Method 3: Post-Stratification ---
    # Uses the SRS split but with post-stratified estimator
    effect_post = 0
    var_post = 0
    for name in pop_weights_full:
        w = pop_weights_full[name]
        if isinstance(name, str):
            name = (name,)
        c_mask = pd.Series([True]*len(ctrl_srs), index=ctrl_srs.index)
        t_mask = pd.Series([True]*len(treat_srs), index=treat_srs.index)
        for col, val in zip(strat_cols, name):
            c_mask &= (ctrl_srs[col] == val)
            t_mask &= (treat_srs[col] == val)
        c_rev = ctrl_srs[c_mask]['revenue']
        t_rev = treat_srs[t_mask]['revenue']
        if len(c_rev) > 0 and len(t_rev) > 0:
            effect_post += w * (t_rev.mean() - c_rev.mean())
            var_post += w**2 * (c_rev.var()/len(c_rev) + t_rev.var()/len(t_rev))
    
    se_post = np.sqrt(var_post)
    t_stat_post = effect_post / se_post if se_post > 0 else 0
    p_post = 2 * (1 - stats.t.cdf(abs(t_stat_post), df=2*n_per_group-2))
    ci_post = (effect_post - 1.96*se_post, effect_post + 1.96*se_post)
    
    results['Post-stratification']['effects'].append(effect_post)
    results['Post-stratification']['rejections'].append(int(p_post < alpha))
    results['Post-stratification']['covers'].append(int(ci_post[0] <= true_effect <= ci_post[1]))

# Convert to arrays
for method in results:
    for key in results[method]:
        results[method][key] = np.array(results[method][key])

In [ ]:
# Print summary statistics
print("Simulation Results (2000 experiments, true effect = $0.50)")
print("=" * 70)
print(f"{'Method':<25} {'Variance':>10} {'Std':>10} {'Power':>10} {'Coverage':>10}")
print("-" * 70)

for method, data in results.items():
    var = np.var(data['effects'])
    std = np.std(data['effects'])
    power = np.mean(data['rejections'])
    coverage = np.mean(data['covers'])
    print(f"{method:<25} {var:>10.5f} {std:>10.4f} {power:>10.1%} {coverage:>10.1%}")

print("\nVariance reduction relative to SRS:")
srs_var = np.var(results['SRS (naive)']['effects'])
for method in ['Pre-stratification', 'Post-stratification']:
    method_var = np.var(results[method]['effects'])
    reduction = (1 - method_var / srs_var) * 100
    print(f"  {method}: {reduction:.1f}%")

In [ ]:
# Visualize the full comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'SRS (naive)': 'steelblue', 'Pre-stratification': 'green', 'Post-stratification': 'darkorange'}

# (1) Distribution of treatment effect estimates
for method, data in results.items():
    axes[0, 0].hist(data['effects'], bins=40, density=True, alpha=0.4, 
                    color=colors[method], label=method)
axes[0, 0].axvline(true_effect, color='red', linestyle='--', linewidth=2, label=f'True effect = ${true_effect}')
axes[0, 0].set_xlabel('Estimated treatment effect ($)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Distribution of Treatment Effect Estimates')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(alpha=0.3)

# (2) Variance comparison (bar chart)
method_names = list(results.keys())
variances = [np.var(results[m]['effects']) for m in method_names]
bars = axes[0, 1].bar(method_names, variances, color=[colors[m] for m in method_names], alpha=0.7)
axes[0, 1].set_ylabel('Variance of estimator')
axes[0, 1].set_title('Estimator Variance by Method')
axes[0, 1].grid(alpha=0.3, axis='y')
for bar, var in zip(bars, variances):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001, 
                    f'{var:.5f}', ha='center', fontsize=10)

# (3) Power comparison
powers = [np.mean(results[m]['rejections']) for m in method_names]
bars = axes[1, 0].bar(method_names, powers, color=[colors[m] for m in method_names], alpha=0.7)
axes[1, 0].axhline(0.80, color='red', linestyle='--', alpha=0.5, label='80% power target')
axes[1, 0].set_ylabel('Statistical Power')
axes[1, 0].set_title('Power (Probability of Detecting True Effect)')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3, axis='y')
for bar, p in zip(bars, powers):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{p:.1%}', ha='center', fontsize=11)

# (4) Effective sample size equivalence
# How many SRS samples would you need to match stratified variance?
srs_variance = np.var(results['SRS (naive)']['effects'])
effective_n = [srs_variance / np.var(results[m]['effects']) * n_per_group for m in method_names]
bars = axes[1, 1].bar(method_names, effective_n, color=[colors[m] for m in method_names], alpha=0.7)
axes[1, 1].axhline(n_per_group, color='gray', linestyle='--', alpha=0.5, label=f'Actual n={n_per_group}')
axes[1, 1].set_ylabel('Effective sample size (per group)')
axes[1, 1].set_title('Effective Sample Size\n(SRS equivalent for same variance)')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3, axis='y')
for bar, n in zip(bars, effective_n):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                    f'{n:.0f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Final quantitative summary
print("\nPractical Interpretation for QuickCart:")
print("=" * 60)

pre_var = np.var(results['Pre-stratification']['effects'])
srs_var = np.var(results['SRS (naive)']['effects'])
reduction_pct = (1 - pre_var / srs_var) * 100
effective_multiplier = srs_var / pre_var

print(f"\nStratifying by device_type + customer_segment:")
print(f"  - Reduces estimator variance by {reduction_pct:.1f}%")
print(f"  - Equivalent to multiplying sample size by {effective_multiplier:.2f}x")
print(f"  - With 500 users/group, stratification gives the precision of ~{effective_multiplier*500:.0f} users/group under SRS")
print(f"  - Power improvement: {np.mean(results['Pre-stratification']['rejections'])*100:.1f}% vs {np.mean(results['SRS (naive)']['rejections'])*100:.1f}% (SRS)")
print(f"\nThis means QuickCart can either:")
print(f"  (a) Run experiments ~{reduction_pct:.0f}% shorter for the same power, or")
print(f"  (b) Detect ~{(1-np.sqrt(1-reduction_pct/100))*100:.0f}% smaller effects with the same duration")

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Random imbalance | Pure random assignment can produce significant covariate imbalances, especially at moderate sample sizes |
| Why imbalance hurts | Imbalanced covariates inflate the variance of the treatment effect estimator (not bias, but noise) |
| Simple Random Sampling | Unbiased but high variance; baseline approach |
| Pre-stratification | Assign within strata to guarantee balance; eliminates between-strata variance; best variance reduction |
| Post-stratification | Adjust estimates after random assignment; nearly as good as pre-strat; useful when you can't stratify beforehand |
| Variance formula | $\text{Var}_{\text{SRS}} - \text{Var}_{\text{strat}} = \frac{2}{n}\sum_h W_h(\mu_h - \mu)^2 \geq 0$ (always helps) |
| Choosing strata | Use variables with high R-squared with the target metric; few strata preferred (4-12 is typical) |
| Practical gain | Variance reduction = R-squared of stratification variable; equivalent to a free increase in sample size |

### What's Next

In **Week 5**, we'll learn about **CUPED** (Controlled-experiment Using Pre-Experiment Data) — a more powerful variance reduction technique that uses pre-experiment outcomes as a continuous covariate, rather than discretizing into strata. CUPED can be viewed as a continuous generalization of stratification.